# Kafka -> protobuf -> JSON

Reads trade events from one Kafka topic and prints them as JSON.

Run Jupyter **in this directory** on the `.venv` interpreter -- the notebook
imports `sdk` from the working directory.

## 1. Schema

Everything comes from `config.yaml` and the Schema Registry: no `.proto` is kept
in the repository, the classes are generated from the latest version of the subject.

Only `kafka-topic` names the stream -- the subject is derived from it
(`...oms-mt5.events.quotes.live01` -> `oms-mt5.events.quotes-value`), so pointing
the notebook at another stream is a one-line change.

In [ ]:
%load_ext autoreload
%autoreload 2

from sdk import Config
from sdk.generate_proto import generate

config = Config.load()
for key, value in config.masked().items():
    print(f"{key:<24} {value}")

# fetches the latest version of the subject, compiles it, imports the classes
schema = generate(config)

print()
print(f"schema id {schema.schema_id}, version {schema.version}, package {schema.package}")
print("messages (addressed by the message-index prefix on the wire):")
for index_path, name in schema.index_map.items():
    indexes = ".".join(str(index) for index in index_path)
    print(f"  [{indexes}] {name}")

print()
print(schema.proto_path.name)
print("-" * 60)
print(schema.proto_path.read_text(encoding="utf-8"))

## 2. Reading

`consume` hands every decoded message to a callback -- a lambda printing to the
console here. `count` is how many to read; `count=0` streams until interrupted.

The client defaults to `auto_offset_reset="latest"`, i.e. only what is produced
after subscribing. The UAT topic is mostly idle, so the cell below replays what
is still retained instead; drop the argument for a realtime read.

In [ ]:
import json

from sdk.kafka_client import KafkaClient

COUNT = 0 # =0 means live consumption until timeout
auto_offset_reset = "latest" # "latest" for live consumption, "earliest" to replay retained messages

with KafkaClient(config, schema=schema, auto_offset_reset=auto_offset_reset) as client:
    print(f"topic      {config.topic}")
    print(f"partitions {client.partitions()}")
    print(f"group.id   {client.group_id}")

    def show(record):
        print()
        print(f"-- p{record.partition}@{record.offset} {record.timestamp:%Y-%m-%d %H:%M:%S} "
              f"key={record.key} schema={record.schema_id} {record.message}")
        print(json.dumps(record.value, indent=2, ensure_ascii=False))

    delivered = client.consume(show, count=COUNT, timeout=10)

print()
print(f"delivered {delivered} of {COUNT}")